# 🤖 Notebook 04 — LLM Semantic Analysis (Preparation)

**Multimodal RCA Engine — Phase 1: Log Extraction & Processing**

This notebook covers:
1. Structuring log sequences for LLM consumption
2. Prompt engineering for Root Cause Analysis (RCA)
3. Building the analysis pipeline architecture
4. Sample Alibaba microservices trace integration
5. Multimodal fusion schema design

> ⚠️ **Note**: This notebook **prepares** the LLM pipeline but does not call LLM APIs yet.
> Actual LLM integration will be implemented in Phase 2.

---

## 4.1 — Setup & Load Processed Data

In [ ]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import PARSED_DIR, PROCESSED_DIR, FIGURES_DIR, RAW_DIR, ensure_directories, print_section

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

ensure_directories()
print("✅ Setup complete!")

In [ ]:
# Load parsed and processed data
hdfs_parsed_path = PARSED_DIR / 'hdfs_parsed.csv'
templates_path = PARSED_DIR / 'hdfs_templates.csv'

if hdfs_parsed_path.exists():
    hdfs_df = pd.read_csv(hdfs_parsed_path)
    print(f"📊 Loaded parsed logs: {hdfs_df.shape}")
else:
    print("❌ Please run Notebooks 02 & 03 first.")

if templates_path.exists():
    templates_df = pd.read_csv(templates_path)
    print(f"📊 Loaded templates: {templates_df.shape}")
    display(templates_df.head())

## 4.2 — LLM Input Structuring

### Design Principle
LLMs need structured, contextualized input. We format log sequences into a clear structure:

```
[SESSION_START] Block: blk_xxxxx
[T1] INFO  component_A: Normal operation message
[T2] WARN  component_B: Warning about resource X  
[T3] ERROR component_C: Failed to complete operation Y
[SESSION_END] Events: 3, Duration: T3-T1
```

In [ ]:
def format_session_for_llm(session_df, block_id, max_events=50):
    """
    Format a log session into LLM-friendly text.
    
    Args:
        session_df: DataFrame of log entries for one session
        block_id: Block ID for this session
        max_events: Maximum number of events to include
    
    Returns:
        str: Formatted session text
    """
    lines = []
    lines.append(f"[SESSION_START] Block: {block_id}")
    lines.append(f"Total events: {len(session_df)}, Showing: {min(len(session_df), max_events)}")
    lines.append("---")
    
    for i, (_, row) in enumerate(session_df.head(max_events).iterrows()):
        level = row.get('level', 'UNKNOWN')
        component = row.get('component', 'unknown')
        content = str(row.get('content', ''))
        event_id = row.get('event_id', '?')
        
        # Truncate long content
        if len(content) > 200:
            content = content[:200] + '...'
        
        lines.append(f"[E{i+1:03d}] [{level:<5s}] [{component}] {content}")
    
    lines.append("---")
    lines.append(f"[SESSION_END] Total: {len(session_df)} events")
    
    return "\n".join(lines)


# Demo: Format a sample session
if 'hdfs_df' in dir() and 'block_id' in hdfs_df.columns:
    sample_blocks = hdfs_df.dropna(subset=['block_id'])['block_id'].unique()[:3]
    
    for block_id in sample_blocks:
        session = hdfs_df[hdfs_df['block_id'] == block_id]
        formatted = format_session_for_llm(session, block_id)
        print_section(f"Formatted Session: {block_id}")
        print(formatted)
        print()

## 4.3 — Prompt Engineering for RCA

We design prompt templates for different analysis tasks:
1. **Anomaly Classification** — Is this session normal or anomalous?
2. **Root Cause Identification** — What caused the anomaly?
3. **Severity Assessment** — How critical is this issue?

In [ ]:
# ============================================
# RCA Prompt Templates
# ============================================

PROMPT_TEMPLATES = {
    "anomaly_classification": {
        "system": """You are an expert Site Reliability Engineer (SRE) specialized in distributed systems 
and cloud infrastructure. You analyze system logs to identify anomalies and failures.

Your task is to classify log sessions as NORMAL or ANOMALOUS based on the patterns, 
error messages, and event sequences you observe.""",
        
        "user": """Analyze the following log session from an HDFS (Hadoop Distributed File System) cluster.
Determine if this session represents normal operation or an anomaly.

## Log Session:
{session_text}

## Analysis Instructions:
1. Examine the sequence of events
2. Look for error patterns, unusual warnings, or missing expected events
3. Consider the component interactions

## Required Output (JSON):
{{
  "classification": "NORMAL" or "ANOMALY",
  "confidence": 0.0 to 1.0,
  "reasoning": "Brief explanation of your classification",
  "key_indicators": ["list", "of", "key", "indicators"]
}}"""
    },
    
    "root_cause_analysis": {
        "system": """You are an expert in Root Cause Analysis (RCA) for distributed systems.
Given a sequence of log events from a failing system, you identify the most likely root cause
of the failure by analyzing event patterns, error propagation, and component dependencies.""",
        
        "user": """Perform Root Cause Analysis on the following anomalous log session.

## Log Session:
{session_text}

## Template Information:
{template_info}

## Analysis Instructions:
1. Identify the initial failure point (first error or unusual event)
2. Trace the error propagation through components
3. Determine the root cause vs. symptoms
4. Suggest remediation actions

## Required Output (JSON):
{{
  "root_cause": "Description of the root cause",
  "root_cause_category": "hardware|software|config|network|resource|unknown",
  "affected_components": ["list of affected components"],
  "error_chain": [
    {{"event": "E001", "description": "Initial trigger"}},
    {{"event": "E005", "description": "Cascading effect"}}
  ],
  "severity": "critical|high|medium|low",
  "remediation": ["Suggested fix 1", "Suggested fix 2"],
  "confidence": 0.0 to 1.0
}}"""
    },
    
    "comparative_analysis": {
        "system": """You are an expert in comparative log analysis for distributed systems.
You compare normal and anomalous log sessions to identify distinguishing patterns.""",
        
        "user": """Compare the following two log sessions — one normal and one anomalous.
Identify the key differences that distinguish the anomalous session.

## Normal Session:
{normal_session}

## Anomalous Session:
{anomaly_session}

## Required Output (JSON):
{{
  "differences": [
    {{"aspect": "...", "normal": "...", "anomaly": "..."}}
  ],
  "key_discriminators": ["list of key differentiating factors"],
  "missing_events": ["events present in normal but absent in anomaly"],
  "extra_events": ["events present in anomaly but absent in normal"]
}}"""
    }
}

# Display templates
for name, template in PROMPT_TEMPLATES.items():
    print_section(f"Prompt Template: {name}")
    print(f"System ({len(template['system'])} chars):")
    print(f"  {template['system'][:150]}...")
    print(f"\nUser ({len(template['user'])} chars):")
    print(f"  {template['user'][:150]}...")
    print()

In [ ]:
# === Build a complete prompt example ===

def build_rca_prompt(session_df, block_id, templates_df=None, prompt_type="root_cause_analysis"):
    """
    Build a complete LLM prompt for RCA analysis.
    
    Returns:
        dict with 'system' and 'user' messages
    """
    template = PROMPT_TEMPLATES[prompt_type]
    
    # Format the session
    session_text = format_session_for_llm(session_df, block_id)
    
    # Format template info if available
    template_info = ""
    if templates_df is not None and 'event_id' in session_df.columns:
        event_ids = session_df['event_id'].dropna().unique()
        relevant_templates = templates_df[templates_df['cluster_id'].isin(event_ids)]
        template_info = relevant_templates.to_string(index=False)
    
    user_message = template['user'].format(
        session_text=session_text,
        template_info=template_info if template_info else "Not available"
    )
    
    return {
        "system": template['system'],
        "user": user_message,
        "metadata": {
            "block_id": block_id,
            "n_events": len(session_df),
            "prompt_type": prompt_type,
            "timestamp": datetime.now().isoformat()
        }
    }

# Demo: Build a full prompt
if 'hdfs_df' in dir() and 'block_id' in hdfs_df.columns:
    sample_block = hdfs_df.dropna(subset=['block_id'])['block_id'].unique()[0]
    session = hdfs_df[hdfs_df['block_id'] == sample_block]
    
    prompt = build_rca_prompt(
        session, sample_block, 
        templates_df=templates_df if 'templates_df' in dir() else None
    )
    
    print_section("Complete RCA Prompt Example")
    print("=== SYSTEM MESSAGE ===")
    print(prompt['system'])
    print("\n=== USER MESSAGE ===")
    print(prompt['user'])
    print("\n=== METADATA ===")
    print(json.dumps(prompt['metadata'], indent=2))

## 4.4 — Multimodal RCA Pipeline Architecture

The following diagram shows how the LLM and VLM components will work together 
in the full multimodal pipeline.

In [ ]:
# === Pipeline Architecture Visualization ===

fig, ax = plt.subplots(figsize=(18, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 10)
ax.axis('off')

# Colors
c_input = '#E3F2FD'
c_process = '#FFF3E0'
c_model = '#E8F5E9'
c_output = '#FCE4EC'
c_border = '#37474F'

def draw_box(ax, x, y, w, h, text, color, fontsize=10, bold=False):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor=c_border, linewidth=2, 
                         joinstyle='round', zorder=2)
    ax.add_patch(rect)
    weight = 'bold' if bold else 'normal'
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, 
            fontweight=weight, zorder=3, wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=c_border, lw=2))

# Title
ax.text(9, 9.5, 'Multimodal RCA Engine — Pipeline Architecture', 
        ha='center', fontsize=18, fontweight='bold', color='#1a237e')

# Layer 1: Data Sources
ax.text(2.5, 8.7, '📥 DATA SOURCES', ha='center', fontsize=11, fontweight='bold', color='#0D47A1')
draw_box(ax, 0.5, 7.8, 2, 0.8, 'System Logs\n(HDFS, BGL, OpenStack)', c_input, 9)
draw_box(ax, 3, 7.8, 2, 0.8, 'Metrics/Dashboards\n(Grafana, CloudWatch)', c_input, 9)
draw_box(ax, 5.5, 7.8, 2, 0.8, 'Trace Data\n(Alibaba Microservices)', c_input, 9)

# Layer 2: Processing
ax.text(4, 6.9, '⚙️ PROCESSING', ha='center', fontsize=11, fontweight='bold', color='#E65100')
draw_box(ax, 0.5, 5.8, 2.3, 1, 'Log Parsing\n(Regex + Drain)\nTemplate Mining', c_process, 9)
draw_box(ax, 3.3, 5.8, 2.3, 1, 'Feature Extraction\nEvent Counting\nTF-IDF', c_process, 9)
draw_box(ax, 6, 5.8, 2.3, 1, 'Visual Processing\nMetric Graphs\nDashboard Capture', c_process, 9)

# Arrows Data -> Processing
draw_arrow(ax, 1.5, 7.8, 1.6, 6.8)
draw_arrow(ax, 4, 7.8, 4.4, 6.8)
draw_arrow(ax, 6.5, 7.8, 7.1, 6.8)

# Layer 3: Models
ax.text(9, 4.8, '🧠 AI MODELS', ha='center', fontsize=11, fontweight='bold', color='#1B5E20')
draw_box(ax, 0.5, 3.5, 3.5, 1.2, '🔤 LLM Module\n(GPT-4 / Gemini / LLaMA)\nSemantic Log Analysis', c_model, 10, True)
draw_box(ax, 5, 3.5, 3.5, 1.2, '🖼️ VLM Module\n(GPT-4V / Gemini Vision)\nVisual Metric Analysis', c_model, 10, True)

# Arrows Processing -> Models
draw_arrow(ax, 1.6, 5.8, 2.2, 4.7)
draw_arrow(ax, 4.4, 5.8, 3.5, 4.7)
draw_arrow(ax, 7.1, 5.8, 6.7, 4.7)

# Layer 4: Fusion
draw_box(ax, 3, 1.8, 5, 1.2, '🔗 MULTIMODAL FUSION\nCross-modal Correlation · Confidence Weighting\nEvidence Aggregation', '#E8EAF6', 11, True)

draw_arrow(ax, 2.2, 3.5, 4.5, 3)
draw_arrow(ax, 6.7, 3.5, 6.5, 3)

# Layer 5: Output
draw_box(ax, 10, 3, 7, 5.5, '', '#FAFAFA', 9)
ax.text(13.5, 8, '📋 OUTPUT', ha='center', fontsize=11, fontweight='bold', color='#B71C1C')
draw_box(ax, 10.5, 6.8, 6, 0.9, '🎯 Root Cause Identification', c_output, 10)
draw_box(ax, 10.5, 5.6, 6, 0.9, '📊 Severity Assessment (Critical→Low)', c_output, 10)
draw_box(ax, 10.5, 4.4, 6, 0.9, '🔗 Error Chain Visualization', c_output, 10)
draw_box(ax, 10.5, 3.2, 6, 0.9, '💡 Remediation Suggestions', c_output, 10)

draw_arrow(ax, 8, 2.4, 10.5, 4.8)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '04_pipeline_architecture.png'), dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: results/figures/04_pipeline_architecture.png")

## 4.5 — Alibaba Microservices Trace: Sample Integration

Here we show how Alibaba's microservices trace data would integrate with our log pipeline.
The trace provides:
- **MSCallGraph**: Call dependencies between microservices (UM → DM)
- **MSRTMCR**: Response time and call rate metrics
- **MSResource**: CPU/memory utilization per container

In [ ]:
# === Simulate Alibaba Microservices Data Structure ===
# (Since the actual data requires downloading ~8GB per hour, we simulate the schema)

# MSCallGraph schema
callgraph_schema = {
    'columns': ['timestamp', 'traceid', 'service', 'rpc_id', 'um', 'uminstanceid', 
                'rpctype', 'interface', 'dm', 'dminstanceid', 'rt'],
    'description': 'Call graph traces between microservices',
    'sample': {
        'timestamp': 115352,
        'traceid': 'T_11560863075',
        'service': 'S_153587416',
        'rpc_id': '0.1',
        'um': 'MS_58845',
        'uminstanceid': 'MS_58845_POD_0',
        'rpctype': 'rpc',
        'interface': 'xOuy6-80Vt',
        'dm': 'MS_71712',
        'dminstanceid': 'MS_71712_POD_244',
        'rt': 2.0
    }
}

# MSRTMCR schema
metrics_schema = {
    'columns': ['timestamp', 'msname', 'msinstanceid', 'nodeid',
                'providerrpc_rt', 'providerrpc_mcr', 'consumerrpc_rt', 'consumerrpc_mcr',
                'writemc_rt', 'writemc_mcr', 'readmc_rt', 'readmc_mcr',
                'writedb_rt', 'writedb_mcr', 'readdb_rt', 'readdb_mcr'],
    'description': 'Response time (RT) and message call rate (MCR) per microservice'
}

print_section("Alibaba Microservices Trace — Data Schemas")
print("📋 MSCallGraph:")
print(f"   Columns: {callgraph_schema['columns']}")
print(f"   Description: {callgraph_schema['description']}")
print(f"\n   Sample entry:")
for k, v in callgraph_schema['sample'].items():
    print(f"     {k}: {v}")

print(f"\n📋 MSRTMCR:")
print(f"   Columns: {metrics_schema['columns']}")
print(f"   Description: {metrics_schema['description']}")

In [ ]:
# === Simulate a microservice call graph for visualization ===

np.random.seed(42)

# Simulate call graph data
n_services = 15
services = [f'MS_{i}' for i in range(n_services)]
n_calls = 30

sim_calls = []
for _ in range(n_calls):
    um = np.random.choice(services)
    dm = np.random.choice([s for s in services if s != um])
    rt = np.random.exponential(10)  # Response time in ms
    is_anomaly = rt > 30  # Slow calls
    sim_calls.append({'um': um, 'dm': dm, 'rt': rt, 'is_slow': is_anomaly})

sim_df = pd.DataFrame(sim_calls)

# Visualize response time distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(sim_df['rt'], bins=30, color='#1976D2', edgecolor='white', alpha=0.8)
ax1.axvline(30, color='red', linestyle='--', linewidth=2, label='Anomaly threshold (30ms)')
ax1.set_xlabel('Response Time (ms)')
ax1.set_ylabel('Frequency')
ax1.set_title('Simulated Microservice Response Times', fontsize=14, fontweight='bold')
ax1.legend()

# Top callers
call_counts = sim_df.groupby('um').size().sort_values(ascending=True)
call_counts.plot(kind='barh', ax=ax2, color='#388E3C', edgecolor='white')
ax2.set_xlabel('Number of Outgoing Calls')
ax2.set_title('Microservice Call Frequency', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '04_microservice_simulation.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4.6 — Multimodal Fusion Schema

This is the JSON schema for the multimodal fusion output — combining LLM and VLM analyses.

In [ ]:
# === Multimodal RCA Output Schema ===

multimodal_rca_schema = {
    "version": "1.0",
    "timestamp": "2026-04-14T21:00:00Z",
    "incident_id": "INC-2026-001",
    
    "llm_analysis": {
        "source": "system_logs",
        "model": "gpt-4 / gemini-1.5-pro",
        "classification": "ANOMALY",
        "confidence": 0.92,
        "root_cause": "DataNode disk failure causing block replication failure",
        "error_chain": [
            {"event": "E001", "desc": "IOException on DataNode disk write"},
            {"event": "E003", "desc": "Block replication failed for blk_xxx"},
            {"event": "E007", "desc": "NameNode marks DataNode as dead"}
        ],
        "affected_components": ["DataNode", "NameNode", "BlockManager"]
    },
    
    "vlm_analysis": {
        "source": "monitoring_dashboards",
        "model": "gpt-4-vision / gemini-1.5-pro-vision",
        "observations": [
            {"metric": "disk_io", "finding": "Spike in disk I/O errors at 14:23"},
            {"metric": "cpu_usage", "finding": "Normal CPU — not a compute issue"},
            {"metric": "network_throughput", "finding": "Drop in replication bandwidth"}
        ],
        "visual_anomaly_score": 0.87
    },
    
    "multimodal_fusion": {
        "strategy": "weighted_evidence_aggregation",
        "final_root_cause": "Physical disk failure on DataNode-7, causing cascading block replication failures",
        "combined_confidence": 0.95,
        "cross_modal_correlation": {
            "log_error_time": "14:23:15",
            "metric_anomaly_time": "14:23:00",
            "temporal_alignment": "strong (15s delta)"
        },
        "severity": "HIGH",
        "remediation": [
            "Immediately decommission DataNode-7",
            "Trigger manual block replication for affected blocks",
            "Replace failed disk and recommission node"
        ]
    }
}

print_section("Multimodal RCA Output Schema")
print(json.dumps(multimodal_rca_schema, indent=2))

# Save schema
schema_path = PROCESSED_DIR / 'multimodal_rca_schema.json'
with open(schema_path, 'w') as f:
    json.dump(multimodal_rca_schema, f, indent=2)
print(f"\n💾 Saved schema: {schema_path}")

## 4.7 — Save Prompt Templates & Pipeline Config

In [ ]:
# Save prompt templates
prompts_path = PROCESSED_DIR / 'rca_prompt_templates.json'
with open(prompts_path, 'w') as f:
    json.dump(PROMPT_TEMPLATES, f, indent=2)
print(f"💾 Saved prompt templates: {prompts_path}")

# Save pipeline configuration
pipeline_config = {
    "version": "1.0",
    "phase": "1 - Log Processing",
    "modules": {
        "log_parsing": {
            "methods": ["regex", "drain"],
            "drain_config": {"depth": 4, "sim_th": 0.4}
        },
        "feature_extraction": {
            "methods": ["event_count", "tfidf", "sequential"],
            "session_grouping": ["block_id", "time_window"]
        },
        "anomaly_detection": {
            "baselines": ["isolation_forest", "lof"],
            "llm_based": "pending_phase_2"
        }
    },
    "datasets": {
        "primary": "HDFS_v1 (with anomaly labels)",
        "secondary": ["BGL", "OpenStack"],
        "trace_data": "Alibaba microservices-v2022"
    },
    "next_phase": {
        "llm_integration": "Call LLM APIs for semantic analysis",
        "vlm_module": "Visual metric analysis with VLM",
        "fusion": "Multimodal data fusion for RCA"
    }
}

config_path = PROCESSED_DIR / 'pipeline_config.json'
with open(config_path, 'w') as f:
    json.dump(pipeline_config, f, indent=2)
print(f"💾 Saved pipeline config: {config_path}")

## 4.8 — Summary & Phase 1 Conclusion

### ✅ Phase 1 Complete — What we built:

| Notebook | Content | Status |
|----------|---------|--------|
| 01 | Data Acquisition & EDA | ✅ |
| 02 | Log Parsing (Regex + Drain) | ✅ |
| 03 | Feature Engineering & Baseline Detection | ✅ |
| 04 | LLM Pipeline Preparation | ✅ |

### 📦 Artifacts Produced:
- Parsed logs (CSV) for HDFS, BGL, OpenStack
- Drain templates mapping
- Event count feature matrices
- Baseline anomaly detection results
- RCA prompt templates (JSON)
- Multimodal fusion schema (JSON)
- Pipeline configuration (JSON)

### ➡️ Phase 2 Roadmap:
1. **LLM Integration**: Connect to GPT-4/Gemini API for semantic log analysis
2. **VLM Module**: Capture and analyze monitoring dashboard screenshots
3. **Multimodal Fusion**: Combine LLM + VLM outputs for comprehensive RCA
4. **Evaluation**: Compare against baselines on labeled datasets
5. **Paper**: Write results for Scopus Q1 publication target